# Fargebilder med CNN og GPU

**Du skal kunne:** forklare et konvolusjonsfilter, lese lagformer og måle om GPU
hjelper. CIFAR-10 er et kjent datasett med små fargebilder av blant annet fly,
biler, fugler og katter. Oppgaven er vanskeligere enn Fashion-MNIST.
Et lite CNN er utgangspunktet; senere trener vi med flere bilder og epoker.

**Utstyr:** GPU anbefales. På CPU kan du begynne med 1000 treningsbilder og én epoke.
Nedlastingen er rundt 170 MB komprimert og bruker mer plass etter utpakking.


## Gjør arbeidsbenken klar

`import` henter verktøy; `as` gir et kortere navn. `Path` håndterer filstier.
Vi finner kursmappen og legger `startkode` i Pythons søkesti, slik at samme kode
virker fra VS Code og JupyterLab. `sett_seed(42)` gir et fast startpunkt for
slumptall; det gjør sammenligning lettere, men lover ikke identiske tall på ulike
maskiner. `velg_enhet()` velger NVIDIA-GPU når CUDA er tilgjengelig, ellers CPU.
En **enhet** er stedet beregningene skjer. Modell og data må ligge på samme enhet.

**Før kjøring:** Gjett om utskriften blir `cpu` eller `cuda`.
**Etter kjøring:** Kontroller at det stemmer med installasjonssjekken.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

# Finn kursmappen både fra en notatbok og fra prosjektroten.
her = Path.cwd().resolve()
kandidater = [her, *her.parents, her / "05-dypere-maskinlaering"]
kursmappe = next(p for p in kandidater if (p / "startkode" / "kursverktoy.py").exists())
if str(kursmappe / "startkode") not in sys.path:
    sys.path.insert(0, str(kursmappe / "startkode"))
from kursverktoy import sett_seed, velg_enhet
sett_seed(42)
enhet = velg_enhet()
print("PyTorch:", torch.__version__, "| beregner på:", enhet)


## Hent og vis data

RGB gir tre kanaler, derfor får en batch formen `[64, 3, 32, 32]`.
Vi lager igjen et fast skille mellom trening og validering; sluttesten brukes ikke.
`permute(1, 2, 0)` flytter kanalaksen sist, slik Matplotlib forventer fargebilder.
**Forutsi:** Kan du selv klassifisere alle de små bildene?


In [ ]:
from torch import nn
from torch.utils.data import Subset, random_split
from torchvision import datasets, transforms
from kursverktoy import lag_lastere, tren, maal
transformasjon = transforms.ToTensor()
alle = datasets.CIFAR10(kursmappe / "data", train=True, download=True,
                       transform=transformasjon)
trening, validering = random_split(alle, [45000, 5000],
                                  generator=torch.Generator().manual_seed(42))
# Start her. GPU-fordypning: 45000 / 5000 og 15–30 epoker.
trening = Subset(trening, range(10000))
validering = Subset(validering, range(2000))
trening_laster, val_laster = lag_lastere(trening, validering, batch=64)
print("Kategorier:", alle.classes)


In [ ]:
bilder, fasit = next(iter(val_laster))
fig, akser = plt.subplots(1, 6, figsize=(12, 3))
for i, akse in enumerate(akser):
    akse.imshow(bilder[i].permute(1, 2, 0))
    akse.set_title(alle.classes[fasit[i]])
    akse.axis("off")
plt.show()


## Et filter flyttes over bildet

Tenk på et lite vindu som leter etter en bestemt kant. Et **konvolusjonslag** bruker
samme lærbare filter på alle steder. Det gir en nyttig antakelse om at et mønster
kan være relevant uansett hvor i bildet det ligger. Filtrene lærer fra data;
vi programmerer dem ikke til å være katteører eller hjul.

`Conv2d(3, 32, 3, padding=1)` kombinerer tre fargekanaler til 32 nye mønsterkart.
Padding legger en kant rundt input slik at høyde og bredde beholdes her.
`MaxPool2d(2)` beholder største verdi i hvert 2 × 2-vindu og halverer sidelengdene.
Etter to slike trinn har vi `[batch, 64, 8, 8]`. Så bruker et MLP disse mønstrene.
**Forutsi:** Hvor mange tall mates inn i første Linear-lag?


In [ ]:
def lag_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(64 * 8 * 8, 128), nn.ReLU(), nn.Linear(128, 10))


In [ ]:
sett_seed(42)
modell = lag_cnn().to(enhet)
probe = torch.zeros(2, 3, 32, 32, device=enhet)
for lag in modell:
    probe = lag(probe)
    print(type(lag).__name__, tuple(probe.shape))
print("Parametre:", sum(p.numel() for p in modell.parameters()))


## Mål CPU og GPU med samme arbeid

Nå isolerer vi selve modellberegningen på en fast, kunstig batch. `time.perf_counter`
måler sekunder. CUDA kan returnere kontroll før beregningen er ferdig, derfor bruker
vi `synchronize` rundt målingen. Vi varmer opp med tre steg før vi tar tiden.
Kopiering, datalesing og validering er utelatt; dette er **ikke** full treningstid.
Første oppvarming kan endre vektene, men form og regnearbeid er like.

`benchmark` er en funksjon: koden inni kjøres når vi kaller den. Vi lager en egen
modell for hver enhet slik at hovedmodellen ikke endres. `del` fjerner referansene
når funksjonen er ferdig; Python frigjør tensorene som ikke lenger brukes.


In [ ]:
from time import perf_counter

def benchmark(sted, batch=64, steg=15):
    sett_seed(42)
    nett = lag_cnn().to(sted)
    x = torch.randn(batch, 3, 32, 32, device=sted)
    y = torch.randint(0, 10, (batch,), device=sted)
    opt = torch.optim.Adam(nett.parameters(), lr=0.001)
    for _ in range(3):
        opt.zero_grad()
        nn.functional.cross_entropy(nett(x), y).backward()
        opt.step()
    if sted.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    start = perf_counter()
    for _ in range(steg):
        opt.zero_grad()
        nn.functional.cross_entropy(nett(x), y).backward()
        opt.step()
    if sted.type == "cuda":
        torch.cuda.synchronize()
    sekunder = perf_counter() - start
    print(sted, f"{batch * steg / sekunder:.0f} bilder/s")
    if sted.type == "cuda":
        print("Topp allokert tensor-minne:", round(torch.cuda.max_memory_allocated()/2**20), "MiB")
    return sekunder

cpu_tid = benchmark(torch.device("cpu"))
if torch.cuda.is_available():
    gpu_tid = benchmark(torch.device("cuda"))
    print(f"Forhold CPU-tid / GPU-tid: {cpu_tid / gpu_tid:.2f}")
else:
    print("Ingen CUDA her. Du kan fortsatt gjøre det lille treningsforsøket.")


## Tren modellen og lagre resultatet

`tren` bruker samme synlige treningsløkkemønster som i notatbok 02, og laster tilbake
beste vekter målt på validering. Tidsutskriften er treningstid per epoke, inkludert
datalesing og flytting, men uten valideringen. Vi lagrer bare vekter og enkle data.
`torch.save` skriver et checkpoint; `weights_only=True` leser tilbake denne typen
innhold. Arkitekturen må gjenskapes før vektene lastes.

`resultater` opprettes om den mangler. Checkpointet gjelder valgt valideringsepoke;
optimizer-tilstanden er ikke lagret, så det er egnet for inferens, ikke nøyaktig
gjenopptakelse av treningen.


In [ ]:
historikk = tren(modell, trening_laster, val_laster, enhet, epoker=5)
resultatmappe = kursmappe / "resultater"
resultatmappe.mkdir(exist_ok=True)
sti = resultatmappe / "cifar-cnn.pt"
torch.save({"modell": {k: v.cpu() for k, v in modell.state_dict().items()},
            "klasser": alle.classes, "seed": 42, "historikk": historikk}, sti)
kopi = lag_cnn().to(enhet)
lagret = torch.load(sti, map_location=enhet, weights_only=True)
kopi.load_state_dict(lagret["modell"])
print("Lastet tilbake fra", sti)
print("Valideringsfeil og treff:", maal(kopi, val_laster, enhet))


## Undersøk lærte mønsterkart

Vi henter output fra første konvolusjonslag og ReLU ved å kalle lag 0 og 1.
`[0, i]` velger bilde 0 og mønsterkart i. Kartene viser aktiveringer, ikke
forklaringer på hele avgjørelsen. Et lyst felt betyr høy verdi i dette kartet.


In [ ]:
modell.eval()
with torch.no_grad():
    kart = modell[1](modell[0](bilder[:1].to(enhet))).cpu()
fig, akser = plt.subplots(1, 6, figsize=(12, 3))
for i, akse in enumerate(akser):
    akse.imshow(kart[0, i], cmap="viridis")
    akse.set_title(f"Kart {i}")
    akse.axis("off")
plt.show()


## GPU-fordypning

På en RTX 3080 med 10 GB er dette lille nettet og batch 64 et forsiktig utgangspunkt,
men faktisk ledig minne avhenger av andre programmer. Øk først datamengden til
45 000/5000 og antall epoker til 15–30. Øk deretter batch til 128 som et eget forsøk.
Logg målte tider; kurset lover ingen bestemt treningstid eller treffandel.

Ved `CUDA out of memory`: reduser batch, restart kernel og kjør igjen. Modeller,
aktiveringer, gradienter og optimizer deler VRAM. `empty_cache()` frigjør ikke
levende tensorer. Se [feilsøking](../FEILSOKING.md).

**Kontroll:** 64 × 8 × 8 = 4096 verdier per bilde inn i MLP-delen. Forklar hvorfor
et større batch ikke automatisk gir bedre modell, selv om gjennomstrømningen øker.

**Neste:** [05-overforingslaering.ipynb](05-overforingslaering.ipynb).
